# Code interpreting with DaoXE + E2B

This example uses [DaoXE](https://daoxe.com) as the LLM backend and E2B's [Code Interpreter](https://github.com/e2b-dev/code-interpreter) to run model-generated Python in a secure sandbox.

DaoXE is a **multi-model, multi-protocol** API gateway. Here we use the **OpenAI-compatible Chat Completions** surface with the official OpenAI Python SDK and a custom base URL:

```text
https://daoxe.com/v1
```

DaoXE also supports other protocols (including **OpenAI Responses** and **Anthropic Messages / Claude protocol**) and multiple model families from its live catalog. This notebook only shows the OpenAI SDK path for easy migration — it is not OpenAI-only or Claude-only.

> **Availability:** DaoXE is **not offered in mainland China**. Use exact model IDs from **your** DaoXE account catalog.


In [ ]:
%pip install openai e2b_code_interpreter python-dotenv


In [ ]:
import os
import json
import base64
from dotenv import load_dotenv
from openai import OpenAI
from e2b_code_interpreter import Sandbox

load_dotenv()

# DaoXE OpenAI-compatible Chat Completions base URL.
# Override with DAOXE_BASE_URL if needed.
DAOXE_BASE_URL = os.getenv("DAOXE_BASE_URL", "https://daoxe.com/v1")
DAOXE_API_KEY = os.getenv("DAOXE_API_KEY")
E2B_API_KEY = os.getenv("E2B_API_KEY")
# Exact model ID from your DaoXE account catalog (do not hardcode a fixed public list).
MODEL_NAME = os.getenv("DAOXE_MODEL")

if not DAOXE_API_KEY:
    raise SystemExit("Set DAOXE_API_KEY (create a key at https://daoxe.com)")
if not E2B_API_KEY:
    raise SystemExit("Set E2B_API_KEY (https://e2b.dev/docs/getting-started/api-key)")
if not MODEL_NAME:
    raise SystemExit("Set DAOXE_MODEL to an exact model ID from your DaoXE account catalog")

SYSTEM_PROMPT = """
## your job & context
you are a python data scientist. you are given tasks to complete and you run python code to solve them.
- the python code runs in jupyter notebook.
- every time you call `execute_python` tool, the python code is executed in a separate cell. it's okay to multiple calls to `execute_python`.
- display visualizations using matplotlib or any other visualization library directly in the notebook. don't worry about saving the visualizations to a file.
- you have access to the internet and can make api requests.
- you also have access to the filesystem and can read/write files.
- you can install any pip package (if it exists) if you need to but the usual packages for data analysis are already preinstalled.
- you can run any python code you want, everything is running in a secure sandbox environment.
"""

tools = [
    {
        "type": "function",
        "function": {
            "name": "execute_python",
            "description": "Execute python code in a Jupyter notebook cell and returns any result, stdout, stderr, display_data, and error.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The python code to execute in a single cell."
                    }
                },
                "required": ["code"]
            }
        }
    }
]

# OpenAI SDK pointed at DaoXE's OpenAI-compatible base URL.
client = OpenAI(api_key=DAOXE_API_KEY, base_url=DAOXE_BASE_URL)
print(f"DaoXE base URL: {DAOXE_BASE_URL}")
print(f"Model ID: {MODEL_NAME}")


In [ ]:
def code_interpret(code_interpreter, code):
    print("Running code interpreter...")

    execution = code_interpreter.run_code(
        code,
        on_stderr=lambda stderr: print("[Code Interpreter]", stderr),
        on_stdout=lambda stdout: print("[Code Interpreter]", stdout),
    )

    if execution.error:
        print("[Code Interpreter ERROR]", execution.error)
        raise Exception(execution.error.value)

    return execution.results


def process_tool_call(code_interpreter, tool_call):
    if tool_call.function.name == "execute_python":
        code = json.loads(tool_call.function.arguments)["code"]
        return code_interpret(code_interpreter, code)
    return []


def chat_with_llm(code_interpreter, user_message):
    print(f"\n{'=' * 50}\nUser Message: {user_message}\n{'=' * 50}")
    print("Waiting for the LLM to respond...")

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        tools=tools,
        tool_choice="auto",
    )

    message = completion.choices[0].message
    print("\nInitial Response:", message)

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        print(f"\nTool Used: {tool_call.function.name}\nTool Input: {tool_call.function.arguments}")
        results = process_tool_call(code_interpreter, tool_call)
        print(f"Tool Result: {results}")
        return results

    raise Exception("Tool calls not found in message content.")


In [ ]:
def main():
    code_interpreter = Sandbox(api_key=E2B_API_KEY)

    try:
        results = chat_with_llm(
            code_interpreter,
            "Generate a realistic sample of adult male heights (cm), plot a histogram with a normal curve overlay, and summarize mean and standard deviation.",
        )
        result = results[0]
        print("Result:", result)
        if hasattr(result, "png") and result.png:
            with open("height_distribution.png", "wb") as f:
                f.write(base64.b64decode(result.png))
            print("Success: Image generated and saved as height_distribution.png")
        else:
            print("No PNG data in the tool result (stdout/text may still be useful).")
    except Exception as error:
        print("An error occurred:", error)
        raise error
    finally:
        code_interpreter.kill()


if __name__ == "__main__":
    main()
